# 3. Validate the published model

Runs the published model over the labelled profiles and compares its
predictions against the hand labels.

In [ ]:
import sys
sys.path.append("../src")

import pickle
import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader

import config
import data
import evaluation
import inference
import model
import plotting
import training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = model.CNN1D()
net.load_state_dict(torch.load(config.CHECKPOINT, map_location=device))
net.eval().to(device)

norm = pickle.load(open(config.NORM_CONSTANTS, "rb"))
profiles = data.merge_to_four_classes(xr.open_dataset(config.LABELLED_DATA))

n_parameters = sum(p.numel() for p in net.parameters())
print(f"model on {device}, {n_parameters:,} parameters")

## Metrics on the validation split

The published model was trained on an 80/20 split with no test set, so the
validation profiles are the ones it never learned from.

In [ ]:
train_ids, val_ids, _ = data.split_profiles(profiles, val_fraction=0.2,
                                            test_fraction=0.0)

val_loader = DataLoader(data.SMPProfileDataset(profiles, val_ids, norm),
                        batch_size=config.BATCH_SIZE,
                        collate_fn=data.pad_and_collate)
metrics = evaluation.evaluate_model(
    net, val_loader, nn.CrossEntropyLoss(ignore_index=config.PADDING_LABEL))

print(f"validation profiles: {len(val_ids)}")
for name in ["loss", "accuracy", "precision", "recall", "f1"]:
    print(f"{name:<10} {metrics[name]:.4f}")

In [ ]:
plotting.plot_confusion_matrix(metrics["confusion_matrix"])
plt.show()

## Per-class accuracy over all labelled profiles

In [ ]:
labelled_set = data.SMPProfileDataset(
    profiles, np.unique(profiles.profile.values), norm)

predictions = inference.predict_profiles(net, labelled_set, device)
true_labels = [labelled_set[i][1].numpy() for i in range(len(labelled_set))]

predicted = np.concatenate(predictions)
truth = np.concatenate(true_labels)

print(f"overall accuracy {(predicted == truth).mean():.4f}\n")
for i, name in enumerate(config.CLASS_NAMES):
    mask = truth == i
    print(f"{name:<20} {(predicted[mask] == i).mean():6.1%}  ({int(mask.sum()):>7,} bins)")

## True against predicted for a single profile

In [ ]:
profile_index = 116

features, labels = labelled_set[profile_index]
plotting.plot_profile_predictions(features, predictions[profile_index],
                                  true_labels=labels.numpy())
plt.show()

## Store predictions alongside the hand labels

Useful for looking at where the model and the labels disagree.

In [ ]:
profiles = inference.add_predictions_to_dataset(
    profiles, predictions, labelled_set.profile_ids)

output_path = config.REPO_ROOT / "outputs" / "labelled_with_predictions.nc"
output_path.parent.mkdir(parents=True, exist_ok=True)
profiles.to_netcdf(output_path, mode="w", format="NETCDF4", engine="netcdf4")
print(f"wrote {output_path}")